# AI Data Query Agent (Colab Notebook)

This notebook contains a complete pipeline that:
1. **LLM-to-Code**: Uses Gemini to translate natural language queries to Pandas code.
2. **Data Processing**: Runs the code dynamically inside a sandbox on a real CSV dataset.
3. **Outputs**: Displays a textual summary, exact computed metrics, and a chart.
4. **Refusal Handling**: Safely refuses out-of-scope questions with `Out of scope`.
5. **Web UI**: Launches a **Gradio** web app with a public sharing link.

### 🔑 Setup Secret
Before running, make sure to add your Gemini API Key to Google Colab Secrets (the key icon on the left panel) with the name `GEMINI_API_KEY`.

In [ ]:
# Install dependencies
!pip install -q gradio google-generativeai pandas plotly

In [ ]:
import os
import json
import pandas as pd
import plotly.express as px
import google.generativeai as genai
import gradio as gr
from google.colab import userdata

# 1. Create sample dataset
sample_data = """Date,Product,Category,Sales,Profit,Quantity,Region
2025-01-15,iPhone 15,Electronics,1200,300,1,North
2025-01-16,MacBook Air,Electronics,2000,500,1,East
2025-01-18,Leather Chair,Furniture,350,80,2,West
2025-01-20,Ergonomic Desk,Furniture,600,150,1,North
2025-02-02,iPhone 15,Electronics,2400,600,2,South
2025-02-10,Java Programming Book,Books,45,15,3,East
2025-02-15,MacBook Air,Electronics,2000,500,1,West
2025-03-01,Leather Chair,Furniture,700,160,4,East
2025-03-05,Data Science Guide,Books,120,40,4,North
2025-03-12,iPad Pro,Electronics,1000,250,1,South
2025-03-20,Running Shoes,Sports,150,45,1,West
2025-04-02,Yoga Mat,Sports,80,25,2,North
2025-04-10,Ergonomic Desk,Furniture,1200,300,2,East
2025-04-15,Data Science Guide,Books,30,10,1,South
2025-04-20,iPhone 15,Electronics,1200,300,1,West
2025-05-01,Running Shoes,Sports,300,90,2,North
2025-05-12,iPad Pro,Electronics,2000,500,2,East
2025-05-25,Leather Chair,Furniture,350,80,1,South
2025-06-01,Yoga Mat,Sports,120,37,3,West"""

with open("sample_data.csv", "w") as f:
    f.write(sample_data)

print("Sample dataset 'sample_data.csv' generated.")

In [ ]:
# 2. Code Pipeline Functions
def query_llm(query, df, api_key):
    if not api_key:
        return {"status": "error", "message": "API Key is missing."}

    genai.configure(api_key=api_key)
    
    schema_info = []
    for col in df.columns:
        dtype = str(df[col].dtype)
        sample_vals = list(df[col].dropna().unique()[:3])
        schema_info.append(f"- Column '{col}' (type: {dtype}). Sample values: {sample_vals}")
    
    schema_str = "\n".join(schema_info)
    df_head = df.head(5).to_string()

    prompt = f"""
You are a data assistant. Your job is to translate a user's natural language query into Python Pandas code to run on a DataFrame named `df`.
You must respond with a raw JSON object and nothing else. No markdown blocks except if you wrap the response in a JSON block.

Here is the schema of the DataFrame `df`:
{schema_str}

Here is a sample of the first 5 rows:
{df_head}

User Query: "{query}"

Instructions:
1. Determine if the query can be answered using the provided DataFrame schema. If NOT (e.g. query is about general knowledge or fields not in the dataset), respond with a JSON where `"status": "out_of_scope"`. Do not guess.
2. If it can be answered, set `"status": "success"`.
3. Provide pandas code in `"pandas_code"`.
   - The DataFrame is loaded as `df`.
   - You MUST compute the final answer and assign it to a variable named `result`.
   - Prepare secondary chart data and store it in `chart_data` (e.g., grouped summary series or dataframe).
4. Provide a clear English explanation in `"explanation"`. Use a placeholder like `{{result}}` where the computed result goes.
5. If a chart is relevant, set `"chart_type"` to `"bar"`, `"line"`, or `"pie"`. Otherwise `"none"`. Specify `"chart_x"` and `"chart_y"` axes.

Response Format (JSON):
{{
  "status": "success" | "out_of_scope",
  "pandas_code": "Pandas code here. E.g. grouped = df.groupby('Category')['Sales'].sum().reset_index(); result = grouped; chart_data = grouped",
  "explanation": "The total sales by category are: \\n {{result}}",
  "chart_type": "bar" | "line" | "none",
  "chart_x": "Category",
  "chart_y": "Sales",
  "chart_title": "Sales by Category"
}}
"""

    model = genai.GenerativeModel('gemini-1.5-flash')
    response = model.generate_content(
        prompt,
        generation_config={"response_mime_type": "application/json"}
    )
    
    try:
        return json.loads(response.text.strip())
    except:
        text = response.text.strip()
        if text.startswith("```json"):
            text = text[7:]
        if text.endswith("```"):
            text = text[:-3]
        return json.loads(text.strip())

def execute_pandas_code(code, df):
    local_vars = {'df': df.copy(), 'pd': pd}
    try:
        exec(code, {}, local_vars)
        return local_vars.get('result', None), local_vars.get('chart_data', None), None
    except Exception as e:
        return None, None, str(e)

In [ ]:
# 3. Gradio Interface Definition
def handle_query(query):
    # Load file
    try:
        df = pd.read_csv("sample_data.csv")
    except Exception as e:
        return "Error loading dataset.", "", "", None

    # Fetch API Key from Colab Secrets
    try:
        api_key = userdata.get('GEMINI_API_KEY')
    except:
        return "Please configure GEMINI_API_KEY in your Colab Secrets (left panel key icon).", "", "", None
        
    if not api_key:
        return "GEMINI_API_KEY is empty in Colab Secrets.", "", "", None
        
    response = query_llm(query, df, api_key)
    
    if response.get("status") == "out_of_scope":
        return "🚫 Out of scope", "N/A", "The query is out of scope of the dataset.", None
        
    pandas_code = response.get("pandas_code", "")
    result, chart_data, error = execute_pandas_code(pandas_code, df)
    
    if error:
        return f"Execution Error: {error}", pandas_code, "Could not execute generated code.", None
        
    explanation = response.get("explanation", "")
    if "{result}" in explanation:
        explanation = explanation.replace("{result}", str(result))
        
    # Render Chart
    fig = None
    chart_type = response.get("chart_type", "none").lower()
    if chart_type != "none" and chart_data is not None:
        chart_x = response.get("chart_x")
        chart_y = response.get("chart_y")
        chart_title = response.get("chart_title", "Visualization")
        
        if isinstance(chart_data, pd.Series):
            chart_data = chart_data.reset_index()
            
        try:
            if chart_type == "bar":
                fig = px.bar(chart_data, x=chart_x, y=chart_y, title=chart_title)
            elif chart_type == "line":
                fig = px.line(chart_data, x=chart_x, y=chart_y, title=chart_title)
            elif chart_type == "pie":
                fig = px.pie(chart_data, names=chart_x, values=chart_y, title=chart_title)
        except Exception as e:
            print(f"Chart render error: {e}")
            
    return (
        explanation,
        pandas_code,
        str(result),
        fig
    )

# Create UI theme
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 📊 AI Data Query Agent")
    gr.Markdown("Type a question to generate code, execute it, and see results. Uses Colab Secret: `GEMINI_API_KEY`.")
    
    with gr.Row():
        query_input = gr.Textbox(placeholder="e.g., Show me the total sales by product", label="Enter Query")
        submit_btn = gr.Button("Submit", variant="primary")
        
    with gr.Row():
        ans_output = gr.Textbox(label="Answer")
        code_output = gr.Code(label="Generated Pandas Code", language="python")
        
    with gr.Row():
        raw_val_output = gr.Textbox(label="Raw Result")
        chart_output = gr.Plot(label="Plotly Chart")
        
    submit_btn.click(handle_query, inputs=[query_input], outputs=[ans_output, code_output, raw_val_output, chart_output])

demo.launch(share=True)